In [1]:
from ucimlrepo import fetch_ucirepo 
  
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from pandas import DataFrame
import numpy as np
from torch.nn import functional as F
from torchinfo import summary
import mlflow
import pandas as pd

# fetch dataset 
iris = fetch_ucirepo(id=53) 
  
# data (as pandas dataframes) 
X = iris.data.features 
y = iris.data.targets 
  
# metadata 
print(iris.metadata) 
  
# variable information 
print(iris.variables) 


{'uci_id': 53, 'name': 'Iris', 'repository_url': 'https://archive.ics.uci.edu/dataset/53/iris', 'data_url': 'https://archive.ics.uci.edu/static/public/53/data.csv', 'abstract': 'A small classic dataset from Fisher, 1936. One of the earliest known datasets used for evaluating classification methods.\n', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Tabular'], 'num_instances': 150, 'num_features': 4, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['class'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1936, 'last_updated': 'Tue Sep 12 2023', 'dataset_doi': '10.24432/C56C76', 'creators': ['R. A. Fisher'], 'intro_paper': {'ID': 191, 'type': 'NATIVE', 'title': 'The Iris data set: In search of the source of virginica', 'authors': 'A. Unwin, K. Kleinman', 'venue': 'Significance, 2021', 'year': 2021, 'journal': 'Significance, 2021', 'DOI': '1740-9713.01589', 'URL': 'https://www.semanticscholar.org

In [2]:
mlflow.set_tracking_uri(uri="http://192.168.100.203:5000/")
mlflow.set_experiment("[P] Classification with Iris Dataset")

dataset = mlflow.data.from_pandas(
   pd.concat([X, y], axis=1), name=iris.metadata["name"], targets=iris.metadata.target_col[0]
)


2025/06/30 22:00:06 INFO mlflow.tracking.fluent: Experiment with name '[P] Classification with Iris Dataset' does not exist. Creating a new experiment.


In [3]:
def preprocess(X: DataFrame, y: DataFrame) -> tuple:
    assert isinstance(X, DataFrame)
    assert isinstance(y, DataFrame)
    assert X.shape[1] >= 1
    assert y.shape[1] == 1

    y_vector: np.ndarray = y.to_numpy().ravel()
    label_encoder = LabelEncoder()
    label_encoder.fit(y_vector)

    mapping = {
        "columns": {idx: col for idx, col in enumerate(X.columns)},
        "labels": {idx: label for idx, label in enumerate(label_encoder.classes_)},
    }

    X_processed: np.ndarray = X.to_numpy()
    y_processed: np.ndarray = label_encoder.transform(y_vector)

    return (
        torch.tensor(X_processed, dtype=torch.float32),
        torch.tensor(y_processed, dtype=torch.long),
        mapping,
    )


In [8]:
class Classifier(nn.Module):
    def __init__(self, input_size=4, hidden_size=16, output_size=3):
        super(Classifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        return self.out(x)


X_p, y_p, mapping = preprocess(X, y)
X_train, X_test, y_train, y_test = train_test_split(
    X_p, y_p, test_size=0.20, random_state=1
)

criterion = nn.CrossEntropyLoss()
model = Classifier()
optimizer = optim.Adam(model.parameters(), lr=0.01)
NUM_EPOCHS = 100

params = {
    "input_size": X_train.shape[1],
    "hidden_size": 16,
    "output_size": len(mapping["labels"]),
    "learning_rate": 0.01,
    "epochs": NUM_EPOCHS,
    "model": Classifier,
    "criterion": criterion,
}

with mlflow.start_run(log_system_metrics=True) as run:
    mlflow.log_params(params)
    mlflow.log_input(dataset, context="training")
    mlflow.set_tag("purpose", "practice")
    mlflow.set_tag("framework", "pytorch")
    mlflow.set_tag("task", "classification")

    for epoch in range(NUM_EPOCHS):
        model.train()
        optimizer.zero_grad()
        outputs = model(X_train)

        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            test_outputs = model(X_test)
            predicted = torch.argmax(test_outputs, dim=1).to(torch.float32)
            accuracy = (predicted == y_test).float().mean().item()


        mlflow.log_metrics(
            {
                "loss": loss.item(),
                "accuracy": accuracy,
            },
            step=epoch,
        )
    mlflow.pytorch.log_model(model, name="Iris Classifier", input_example=X_train.numpy()[0])

2025/06/30 22:02:09 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2025/06/30 22:02:09 WARNING mlflow.system_metrics.metrics.gpu_monitor: Encountered error Not Supported when trying to collect GPU power usage metrics.


2025/06/30 22:02:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/06/30 22:02:14 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2025/06/30 22:02:14 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run nosy-carp-42 at: http://192.168.100.203:5000/#/experiments/3/runs/b4ec1799dd4a42dba3bc52af9fc605e8
🧪 View experiment at: http://192.168.100.203:5000/#/experiments/3
